# Lesson 5b — Streaming churn predictor, executable line by line

This notebook is a runnable version of the [Lesson 5b walkthrough](../05b_walkthrough.md).

Same PRAGMA recipe as Lesson 5, but now applied to a more realistic
problem: predicting which streaming-service users are about to cancel.

We'll do the full foundation-model workflow:

1. Generate synthetic event data — engaged vs churning users.
2. **Pre-train** a Transformer with fill-in-the-blank — no labels used.
3. **Freeze** the encoder, add a tiny classifier head.
4. **Train** only the head on a small labelled set.
5. **Compare** to a baseline that doesn't pre-train.

## 🧰 Lesson reference legend

- **L1** — the 5-line training loop
- **L1b** — architecture vs. training
- **L1c** — gradient descent details
- **L2** — tokens & embeddings
- **L3** — attention
- **L4** — masked language modelling
- **L5** — `pragma_mini.py`


## 0 — Imports and seeds

In [ ]:
import random                                # Python stdlib RNG (used by data generators).
import copy                                  # For copy.deepcopy to snapshot the encoder.
import torch                                 # PyTorch core.
import torch.nn as nn                        # Neural network building blocks.

torch.manual_seed(7)                         # Reproducible PyTorch RNG (different seed = different data).
random.seed(7)                               # Reproducible Python RNG.
torch.set_printoptions(precision=3, sci_mode=False)     # Clean tensor printing.

## 1 — Vocabulary (L2 + L5)

Same key-value vocabulary structure as `pragma_mini.py`, with more keys
and richer values:

- **Keys**: action, genre, time, duration
- **Values per key**: 4-5 choices each

Total vocab: ~23 tokens.


In [ ]:
KEYS     = ["action", "genre", "time", "duration"]      # 4 field names per event.
ACTIONS  = ["start", "finish", "skip", "pause", "browse"]   # Action values.
GENRES   = ["comedy", "drama", "action", "documentary", "kids"]
TIMES    = ["morning", "afternoon", "evening", "night"]
DURATION = ["short", "medium", "long"]

PAD, MASK = "<pad>", "<mask>"
vocab  = [PAD, MASK] + KEYS + ACTIONS + GENRES + TIMES + DURATION    # Combined vocab.
tok2id = {t: i for i, t in enumerate(vocab)}             # token → integer id lookup.
V      = len(vocab)                                      # Total vocab size.

print(f"vocab size: {V}")
print(f"vocab: {vocab}")

## 2 — Synthetic data (engaged vs churning)

Two user "types" with distinct behaviour distributions:

- **ENGAGED**: mostly *finishes* episodes, varied genres, *long* sessions, watches in the *evening*.
- **CHURNING**: lots of *skipping*, narrow genre, *short* sessions, mostly at *night*.

The model has to discover these patterns from the event sequences alone.


In [ ]:
def engaged_event():                        # Sample one event for an engaged user.
    a = random.choices(ACTIONS, weights=[2, 8, 1, 1, 1])[0]      # Mostly 'finish'.
    g = random.choice(GENRES)                # Uniform across genres (varied tastes).
    t = random.choices(TIMES,   weights=[1, 2, 5, 2])[0]         # Mostly evening.
    d = random.choices(DURATION, weights=[1, 3, 6])[0]           # Mostly long sessions.
    return [("action", a), ("genre", g), ("time", t), ("duration", d)]

def churning_event(narrow_genre):           # Sample one event for a churning user.
    a = random.choices(ACTIONS, weights=[3, 1, 6, 1, 4])[0]      # Mostly 'skip'.
    g = random.choices([narrow_genre] * 6 + GENRES, k=1)[0]      # Narrow genre.
    t = random.choices(TIMES,   weights=[2, 3, 2, 5])[0]         # Mostly night.
    d = random.choices(DURATION, weights=[7, 2, 1])[0]           # Mostly short sessions.
    return [("action", a), ("genre", g), ("time", t), ("duration", d)]

def make_user(churning):                    # Build a 15-event user history.
    if churning:
        narrow = random.choice(GENRES)      # Pick the genre this user obsesses over.
        return [churning_event(narrow) for _ in range(15)]
    return [engaged_event() for _ in range(15)]

# Show one of each type
print("ENGAGED user (first 5 events):")
for e in make_user(False)[:5]:
    print("  ", e)
print("\nCHURNING user (first 5 events):")
for e in make_user(True)[:5]:
    print("  ", e)

## 3 — Build the dataset

2000 users, 10% churning. Each user gets encoded as a flat sequence of
tokens (15 events × 8 tokens/event = 120 tokens per user).


In [ ]:
def encode_event(event):                    # Flatten one event to token ids.
    ids = []
    for k, v in event:                      # Iterate (key, value) pairs.
        ids.append(tok2id[k])               # Key id.
        ids.append(tok2id[v])               # Value id.
    return ids

def encode_user(events):                    # Flatten a whole user history.
    return [tok for e in events for tok in encode_event(e)]    # Concatenate all event tokens.

N_USERS, CHURN_RATE = 2000, 0.10            # 2000 users, 10% will be marked as churning.

users, labels = [], []
for _ in range(N_USERS):
    churn = random.random() < CHURN_RATE    # Coin flip decides churn status.
    users.append(make_user(churn))          # Generate that user's events.
    labels.append(1 if churn else 0)        # 1 = churn, 0 = engaged.

X = torch.tensor([encode_user(u) for u in users], dtype=torch.long)    # (N_USERS, 120) tokens.
y = torch.tensor(labels, dtype=torch.long)                              # (N_USERS,) binary labels.

print(f"shape X: {tuple(X.shape)}")
print(f"labels: {int(y.sum())} churning, {int((y == 0).sum())} engaged")

## 4 — Architecture (L1b + L2 + L3)

Three components:

| Piece | Lesson | What it does |
|-------|--------|--------------|
| `Encoder` | L2 + L3 | Embedding + position + Transformer encoder. The PRAGMA backbone. |
| `MLMHead` | L4 | Predicts a masked token. Used during pre-training. |
| `ChurnHead` | NEW | Pools the encoder output, predicts engaged/churn. Used downstream. |

Identical encoder architecture to `pragma_mini.py`. The new bit is
`ChurnHead` — it averages the encoder's per-position outputs into one
vector for the whole user, then projects to 2 churn classes.


In [ ]:
D_MODEL, N_HEADS, N_LAYERS = 32, 2, 2        # Model dimensions: 32-d vectors, 2 heads, 2 layers.

class Encoder(nn.Module):                   # The shared backbone: embed + position + Transformer.
    def __init__(self, V, d=D_MODEL, heads=N_HEADS, layers=N_LAYERS, max_len=128):
        super().__init__()
        self.emb = nn.Embedding(V, d)        # Token embedding table.
        self.pos = nn.Embedding(max_len, d)  # Position embeddings (supports up to 128 positions).
        layer    = nn.TransformerEncoderLayer(d, heads, d * 2, batch_first=True)    # One block.
        self.enc = nn.TransformerEncoder(layer, layers)     # Stack `layers` copies of the block.
    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device)    # [0, 1, ..., L-1].
        return self.enc(self.emb(x) + self.pos(positions))      # Embed + add pos, then encode.

class MLMHead(nn.Module):                   # Used during PRE-TRAINING (predict masked tokens).
    def __init__(self, V, d=D_MODEL):
        super().__init__()
        self.proj = nn.Linear(d, V)         # Project each hidden vector to vocab scores.
    def forward(self, h):
        return self.proj(h)

class ChurnHead(nn.Module):                 # Used during DOWNSTREAM (predict churn class).
    def __init__(self, d=D_MODEL):
        super().__init__()
        self.proj = nn.Linear(d, 2)         # 2 classes: engaged (0) or churning (1).
    def forward(self, h):
        pooled = h.mean(dim=1)              # Mean-pool across positions to a single per-user vector.
        return self.proj(pooled)            # 2 logits per user.

enc_check = Encoder(V)
print(f"Encoder params: {sum(p.numel() for p in enc_check.parameters()):,}")
print(f"MLMHead weights: {sum(p.numel() for p in MLMHead(V).parameters()):,}")
print(f"ChurnHead weights: {sum(p.numel() for p in ChurnHead().parameters())}")

## 5 — Pre-train via masked language modelling (L4)

For each batch, hide ~20% of the value tokens (never key tokens — those
stay visible as a hint about what type of thing to predict). Train the
model to fill them in.

This is **self-supervised** — labels are made on the fly from the data
itself. The churn labels are NEVER used here.


In [ ]:
KEY_IDS = torch.tensor([tok2id[k] for k in KEYS])    # Tensor of all KEY token ids (never masked).

def mlm_mask(X_batch, p=0.20):              # Mask ~20% of VALUE tokens (never key tokens).
    X = X_batch.clone()                     # Don't mutate the input.
    y = torch.full_like(X, -100)            # Start labels as -100 (ignore everywhere).
    is_value = ~torch.isin(X, KEY_IDS)      # Boolean mask: True where token is a VALUE.
    pick = (torch.rand_like(X, dtype=torch.float) < p) & is_value    # Random subset of values.
    y[pick] = X[pick]                       # Remember truth at picked positions.
    X[pick] = tok2id[MASK]                  # Replace those positions with <mask>.
    return X, y

encoder  = Encoder(V)                       # Fresh encoder (random init).
mlm_head = MLMHead(V)                       # Fresh MLM head for pre-training.
opt      = torch.optim.AdamW(
    list(encoder.parameters()) + list(mlm_head.parameters()), lr=3e-3)     # Train both.
loss_fn  = nn.CrossEntropyLoss(ignore_index=-100)        # CE that skips -100 labels.

print("Pre-training (2000 steps)...")
for step in range(2000):
    idx       = torch.randint(0, N_USERS, (64,))         # Random batch of 64 user ids.
    xb, yb    = mlm_mask(X[idx])                          # Mask their value tokens.
    h         = encoder(xb)                               # Forward through encoder.
    logits    = mlm_head(h)                               # Project to vocab scores.
    loss      = loss_fn(logits.reshape(-1, V), yb.reshape(-1))    # Cross-entropy on masked.
    opt.zero_grad(); loss.backward(); opt.step()          # Standard 3-line update.
    if step % 400 == 0:
        print(f"  step {step:4d}   MLM loss {loss.item():.3f}")

# Save the pre-trained encoder
pretrained_encoder = copy.deepcopy(encoder)              # Deep copy so subsequent training
                                                         # doesn't mutate this snapshot.
print("\nDone. Encoder weights cached as `pretrained_encoder`.")

## 6 — Train and test split

In [ ]:
perm  = torch.randperm(N_USERS)             # Random permutation of all user indices.
split = int(N_USERS * 0.8)                  # 80% train, 20% test.
tr_idx, te_idx = perm[:split], perm[split:] # Index lists for train/test.
X_te, y_te     = X[te_idx], y[te_idx]       # Test set tensors.
print(f"Test set: {len(te_idx)} users  ({int(y_te.sum())} churning)")

## 7 — Downstream task: predict churn

Two modes to compare:

- **A. Frozen pre-trained encoder + classifier head.** The encoder's
  weights stay fixed (`requires_grad=False`); only the small `ChurnHead`
  gets trained. This is the foundation-model recipe.
- **B. Random-init encoder, train everything end-to-end.** No
  pre-training. The baseline.

We'll run both at 4 label-count settings: 20, 50, 200, 1000.


In [ ]:
def freeze(mod):                            # Freeze a module's params (no gradient updates).
    for p in mod.parameters(): p.requires_grad = False      # Stop tracking gradients.
    mod.eval()                              # Disable dropout / batchnorm running stats.

def train_classifier(encoder, X_tr, y_tr, epochs=200, freeze_encoder=True):
    head = ChurnHead()                      # Fresh classifier head each call.
    if freeze_encoder:
        freeze(encoder)                     # Pre-training recipe: encoder is fixed.
        params = list(head.parameters())    # Only train the head.
    else:
        params = list(encoder.parameters()) + list(head.parameters())   # Train EVERYTHING.
    opt = torch.optim.AdamW(params, lr=3e-3)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        h      = encoder(X_tr)              # Encode training inputs.
        logits = head(h)                    # 2-class logits.
        loss   = loss_fn(logits, y_tr)      # CE against churn/engaged labels.
        opt.zero_grad(); loss.backward(); opt.step()
    head.eval()
    with torch.no_grad():
        logits = head(encoder(X_te))        # Predict on test set.
        pred   = logits.argmax(-1)          # argmax → predicted class.
        acc    = (pred == y_te).float().mean().item()
        churn  = (y_te == 1)                # Mask of actual churners.
        recall = (pred[churn] == 1).float().mean().item() if churn.sum() > 0 else float("nan")
    return acc, recall

print(f"{'labels':>7} | {'pretrained acc':>14}  {'pretrained recall':>17} | "
      f"{'baseline acc':>12}  {'baseline recall':>15}")
print("-" * 84)
for n_labels in [20, 50, 200, 1000]:        # Try 4 different labelled-set sizes.
    sub = tr_idx[:n_labels]                 # Take first n labelled examples.
    X_tr, y_tr = X[sub], y[sub]

    enc_a = copy.deepcopy(pretrained_encoder)       # Fresh copy of pre-trained encoder.
    acc_a, rec_a = train_classifier(enc_a, X_tr, y_tr, freeze_encoder=True)  # Recipe A.

    torch.manual_seed(n_labels)             # Different seed per run for variety.
    enc_b = Encoder(V)                      # Random-init encoder.
    acc_b, rec_b = train_classifier(enc_b, X_tr, y_tr, freeze_encoder=False) # Recipe B.

    print(f"{n_labels:>7} | {acc_a:>14.3f}  {rec_a:>17.3f} | {acc_b:>12.3f}  {rec_b:>15.3f}")

## 8 — Interpreting the results

What to look for in the table above:

- **20 labels: both fail.** Too few examples of the rare class. Accuracy ≈ 0.9 is the "always-predict-engaged" baseline (since ~90% of users are engaged).
- **50 labels: pre-training wins decisively.** The frozen probe usually catches close to 100% of churners. The random-init baseline lags substantially behind on recall.
- **200 labels: pre-training still ahead.**
- **1000 labels: they converge.** Plenty of data — even the random-init baseline can learn the pattern end-to-end.

> 🔑 **The pre-trained encoder is most valuable when labelled data is
> scarce.** This is the entire foundation-model pitch — and the reason
> Revolut built PRAGMA. Labels are expensive; raw events are free.

## Things to try

1. **Make the churn signal harder.** Edit `churning_event` to look more like `engaged_event`. Re-run both pre-train and classifier. Does pre-training still help?
2. **Smaller pre-train.** Reduce pre-training steps to 200. Does the probe still beat the baseline?
3. **Look at the pre-trained embeddings.** Run `pretrained_encoder.emb.weight[tok2id["finish"]]` and compare to `tok2id["skip"]`. Are "engaged-y" and "churn-y" actions clustered?

## What's next

You're ready for **[Lesson 6 — Capstone](../06_capstone.md)**: same recipe, fraud detection, you build it yourself.
